<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!git clone https://github.com/yaranoun/ML-Tech.git

fatal: destination path 'ML-Tech' already exists and is not an empty directory.


In [13]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 950 bytes | 475.00 KiB/s, done.
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
   78b52c4..fa84563  main       -> origin/main
Updating 78b52c4..fa84563
Fast-forward
 data/processed/chunks.json | 101 ++++++++++++++++++++++++++++++++++++++-------
 1 file changed, 87 insertions(+), 14 deletions(-)


In [14]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 17 chunks
{'document': 'Personal attendance required.txt', 'title': 'Personal attendance required', 'url': 'https://www.general-security.gov.lb/en/posts/73', 'category': 'Personal attendance required', 'keywords': 'Lebanese citizens, minors, exemption from attendance, exemption from fees', 'section': 'Personal attendance required', 'text': 'Lebanese citizens that request a new passport should show up personally at the competent regional center of general security, according to their place of residence, having in hand an application that’s been filled, and certified by the competent mayor.\nMinors aged 7 years or younger have to accompany their parents to the mayor’s office, but don’t have to show up at the general security center. Both parents should sign a letter of consent at the mayor’s office, and convey their request to the general security. One of the parents can go on his own to the general security office, if the other parent signed the letter at the mayor’s office.\nMin

In [15]:
!pip install -q sentence-transformers faiss-cpu

In [16]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [17]:
texts = [chunk["text"] for chunk in chunks]
passages = ["passage: " + text for text in texts]

In [18]:
embeddings = model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(17, 768)


In [19]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [20]:
question="What documents are required for a biometric passport?"
question_embedding = model.encode(["query:" + question], normalize_embeddings=True)

In [21]:
k = 3

scores, indices = index.search(question_embedding, k)
for rank, idx in enumerate(indices[0]):
    print("=" * 80)
    print("Rank:", rank + 1)
    print("Score:", scores[0][rank])
    print("Document:", chunks[idx]["document"])
    print("Section:", chunks[idx]["section"])
    print("Text:")
    print(chunks[idx]["text"])

Rank: 1
Score: 0.858559
Document: Biometric Passport.txt
Section: Requested documents
Text:
The adequate application for passports format A4 (10 years) issued by the competent mayor according to the place of residence.
Lebanese ID card OR/AND an extract of civil status (whether the Lebanese citizen is applying for the 1st time for a biometric passport or not). Follow this link for more information: https://www.general-security.gov.lb/ar/posts/408
A new colored photo ID photo on a white background, 4.5 x 3.5, on which the name of the individual appears, as well as the number and place of registered residence, signed and certified by the mayor.
The old passport if the latter is available, as well as a copy of the pages that are not empty.
The fees related to this application.
When it comes to members of the general security, whether active or retired, and the applications sent by their families, they can hand out only one document of identification (identity card, or extract of individua